# Path / Labs report pipeline - one notebook, no PDF files

For every **active** user in `credentials.csv` (`isActive = 1`) the notebook logs in once and downloads each tab flagged `1`
(`Pending_plan_completion`, `Completed`, `Unresolved`). Every report is produced **in memory as PDF bytes**, parsed straight
into rows (column positions are read from the PDF's own header row) and discarded - **no PDF is saved anywhere** unless you
switch `save_pdfs` on.

| Step | What happens |
|---|---|
| 1 | Chromium is installed on the driver (your no-root setup cell). |
| 2 | The browser automation runs as an embedded worker process (Playwright cannot run inside a notebook's event loop) and streams each report back as bytes. |
| 3 | Each report is parsed on arrival; rows go into one Spark DataFrame (`pathology_df`) with a `category` column. |
| 4 | CSVs (optional) are written to `downloads/<run>/pending/` and `.../completed/` with suffix **`_pend`** / **`_comp`** / **`_unres`**; optional Delta tables get the same suffix. |
| 5 | Per category, every report is checked against the *"Rows a-b of N results"* line printed inside it. |

**Needs:** classic compute, Dedicated access mode, outbound internet (Chromium download + the target site).
`credentials.csv` (input only) sits in `base_dir`: `Path, Username, Password, provider, patient, location, entrydate, Pending_plan_completion, Completed, Unresolved, isActive`.

In [0]:
%pip install playwright pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.8 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/48.5 MB ? eta -:--:--
   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/48.5 MB 138.5 MB/s eta 0:00:01
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/48.5 MB 125.5 MB/s eta 0:00:01
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/48.5 MB 110.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/48.5 MB 110.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━ 20.4/48.5 MB 110.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 25.1/48.5 MB 122.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 29.8/48.5 MB 137.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 33.7/48.5 MB 122.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 40.4/48.5 MB 154.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 45.5/48.5 MB 172

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("base_dir", "/Workspace/Shared/HANDSON SANDBOX", "Base folder")
dbutils.widgets.text("credentials_file", "", "credentials.csv (blank = <base_dir>/credentials.csv)")
dbutils.widgets.text("site_url", "https://musical-biscuit-f1d2b1.netlify.app/", "Site URL (for relative Path values)")
dbutils.widgets.text("users", "", "Only these usernames (comma separated, blank = all active)")
dbutils.widgets.text("limit", "", "Only the first N active users (blank = all)")
dbutils.widgets.dropdown("screenshots", "failures", ["failures", "all", "none"], "Screenshots (saved to <base_dir>/screenshots)")
dbutils.widgets.dropdown("save_csv", "no", ["yes", "no"], "Write CSVs to downloads/<run>/<category>/ ?")
dbutils.widgets.dropdown("save_pdfs", "no", ["no", "yes"], "Also keep the PDFs (audit copy)?")
dbutils.widgets.text("output_table", "", "Delta table base name (optional, e.g. us_scdb_phi_dev.comm_bronze.pathology_results)")

BASE_DIR = dbutils.widgets.get("base_dir").strip().rstrip("/")
CREDENTIALS_FILE = dbutils.widgets.get("credentials_file").strip() or f"{BASE_DIR}/credentials.csv"
SITE_URL = dbutils.widgets.get("site_url").strip()
USERS = dbutils.widgets.get("users").strip()
LIMIT = dbutils.widgets.get("limit").strip()
SNAPSHOTS = dbutils.widgets.get("screenshots")
SAVE_CSV = dbutils.widgets.get("save_csv") == "yes"
SAVE_PDFS = dbutils.widgets.get("save_pdfs") == "yes"
OUTPUT_TABLE = dbutils.widgets.get("output_table").strip()
RUN_AUTOMATION = True          # the install cell below only runs when this is True

In [0]:
import os
import subprocess
import sys
from datetime import datetime

DOWNLOAD_DIR = os.path.join(BASE_DIR, "downloads")


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]


if RUN_AUTOMATION:
    # --with-deps also installs the Linux libraries Chromium needs (needs root: dedicated classic compute).
    rc, out = _run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])
    if rc != 0:
        print("install --with-deps failed, retrying without system dependencies...\n", out)
        rc, out = _run([sys.executable, "-m", "playwright", "install", "chromium"])
    print("Chromium download:", "OK" if rc == 0 else "FAILED")
    if rc != 0:
        print(out)
        raise RuntimeError(
            "Could not install Chromium. Use a classic, dedicated-access cluster with internet access "
            "(or install the browser with a cluster init script)."
        )

# ---- Shared libraries without root -------------------------------------------------------------
# The notebook user is not root, so `--with-deps` cannot apt-install. Any library that `ldd` reports as
# missing is downloaded with `apt-get download` (user-owned apt dirs) and extracted with `dpkg-deb -x`.
PW_LIB_DIR = "/tmp/pwlibs"
PW_LD_PATH = f"{PW_LIB_DIR}/usr/lib/x86_64-linux-gnu:{PW_LIB_DIR}/lib/x86_64-linux-gnu"
os.environ["LD_LIBRARY_PATH"] = PW_LD_PATH + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")

# soname -> Ubuntu 24.04 (noble) package
SONAME_TO_PKG = {
    "libatk-1.0.so.0": "libatk1.0-0t64", "libatk-bridge-2.0.so.0": "libatk-bridge2.0-0t64",
    "libatspi.so.0": "libatspi2.0-0t64", "libXdamage.so.1": "libxdamage1",
    "libXcomposite.so.1": "libxcomposite1", "libXfixes.so.3": "libxfixes3", "libXrandr.so.2": "libxrandr2",
    "libxkbcommon.so.0": "libxkbcommon0", "libgbm.so.1": "libgbm1", "libasound.so.2": "libasound2t64",
    "libcups.so.2": "libcups2t64", "libnss3.so": "libnss3", "libnssutil3.so": "libnss3",
    "libsmime3.so": "libnss3", "libnspr4.so": "libnspr4", "libdrm.so.2": "libdrm2",
    "libpango-1.0.so.0": "libpango-1.0-0", "libcairo.so.2": "libcairo2", "libXext.so.6": "libxext6",
    "libX11.so.6": "libx11-6", "libxcb.so.1": "libxcb1", "libdbus-1.so.3": "libdbus-1-3",
    "libexpat.so.1": "libexpat1", "libgio-2.0.so.0": "libglib2.0-0t64", "libglib-2.0.so.0": "libglib2.0-0t64",
    "libgobject-2.0.so.0": "libglib2.0-0t64", "libudev.so.1": "libudev1", "libXi.so.6": "libxi6",
    "libwayland-server.so.0": "libwayland-server0",
}


def chromium_binaries():
    root = os.environ.get("PLAYWRIGHT_BROWSERS_PATH") or os.path.expanduser("~/.cache/ms-playwright")
    pats = ["chromium_headless_shell-*/chrome-headless-shell-linux64/chrome-headless-shell",
            "chromium-*/chrome-linux*/chrome"]
    return sorted(p for pat in pats for p in glob.glob(os.path.join(root, pat)))


def missing_libs():
    missing = set()
    for b in chromium_binaries():
        out = subprocess.run(["ldd", b], capture_output=True, text=True, env=os.environ).stdout
        missing |= {l.split("=>")[0].strip() for l in out.splitlines() if "not found" in l}
    return sorted(missing)


def install_libs_userspace(sonames):
    pkgs = sorted({SONAME_TO_PKG[s] for s in sonames if s in SONAME_TO_PKG})
    unknown = [s for s in sonames if s not in SONAME_TO_PKG]
    if unknown:
        print("No package mapping for:", unknown, "- add them to SONAME_TO_PKG")
    if not pkgs:
        return
    apt, debs = "/tmp/apt", "/tmp/pwdebs"
    for d in (f"{apt}/lists/partial", f"{apt}/cache/archives/partial", debs, PW_LIB_DIR):
        os.makedirs(d, exist_ok=True)
    opts = ["-o", f"Dir::State::Lists={apt}/lists", "-o", f"Dir::Cache={apt}/cache",
            "-o", "Debug::NoLocking=1", "-o", f"APT::Sandbox::User={getpass.getuser()}"]
    if not os.listdir(f"{apt}/lists") or os.listdir(f"{apt}/lists") == ["partial"]:
        subprocess.run(["apt-get", *opts, "update"], capture_output=True, text=True)
    r = subprocess.run(["apt-get", *opts, "download", *pkgs], cwd=debs, capture_output=True, text=True)
    print("apt-get download", " ".join(pkgs), "->", "OK" if r.returncode == 0 else r.stderr[-800:])
    for deb in glob.glob(os.path.join(debs, "*.deb")):
        subprocess.run(["dpkg-deb", "-x", deb, PW_LIB_DIR], check=True)


LAUNCH_TEST = """
from playwright.sync_api import sync_playwright
with sync_playwright() as p:
    b = p.chromium.launch(args=['--no-sandbox', '--disable-dev-shm-usage'])
    pg = b.new_page(); pg.set_content('<h1>ok</h1>'); pg.pdf()
    print('Chromium OK', b.version); b.close()
"""

if RUN_AUTOMATION:
    import getpass
    import glob

    for attempt in range(3):              # libraries can pull in further missing libraries
        missing = missing_libs()
        if not missing:
            break
        print("Missing shared libraries:", missing)
        install_libs_userspace(missing)

    # Real launch test in a subprocess (the sync API cannot run inside the notebook event loop).
    r = subprocess.run([sys.executable, "-c", LAUNCH_TEST], capture_output=True, text=True, env=os.environ)
    if r.returncode != 0 or "Chromium OK" not in r.stdout:
        still = missing_libs()
        raise RuntimeError(
            "Chromium was downloaded but cannot start.\n"
            f"ldd 'not found': {still or 'none'}\n\nlaunch stderr:\n{r.stderr[-2000:]}"
        )
    print(r.stdout.strip(), "| LD_LIBRARY_PATH includes", PW_LIB_DIR)
else:
    print("Skipping browser install (run_automation = no)")

install --with-deps failed, retrying without system dependencies...
 Installing dependencies...
Switching to root user to install dependencies...
Failed to install browsers
Error: Installation process exited with code: 1
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required

Chromium download: OK
Missing shared libraries: ['libXcomposite.so.1', 'libXdamage.so.1', 'libXfixes.so.3', 'libatk-1.0.so.0', 'libatk-bridge-2.0.so.0', 'libatspi.so.0', 'libcups.so.2', 'libxkbcommon.so.0']
apt-get download libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libcups2t64 libxcomposite1 libxdamage1 libxfixes3 libxkbcommon0 -> OK
Missing shared libraries: ['libavahi-client.so.3', 'libavahi-common.so.3']
No package mapping for: ['libavahi-client.so.3', 'libavahi-common.so.3'] - add them to SONAME_TO_PKG
Missing shared libraries: ['libavahi-client.so.3', 'libavahi-common.so.3']
No package mapp

In [0]:
WORKER_SOURCE = r'''"""
Browser worker (runs as a separate process, started by the notebook).

For every ACTIVE user in credentials.csv: login once, open Path / Labs > Path, then for each
flagged tab (Pending Plan Completion, Completed, Unresolved): fill the filters, Apply Filter and
print every results page. Each report is produced IN MEMORY (bytes) and streamed to the notebook
over stdout - no PDF file is written anywhere. Protocol lines (everything else is a plain log):

    @@PDF@@  {json: username, category, folder, page_no, suggested, captured_at, n_records, declared, pdf_b64}
    @@USER@@ {json: username, status, errors}
    @@DONE@@ {json: users, failed}

Environment: RPA_BASE_DIR, RPA_CREDENTIALS, RPA_SITE_URL, RPA_USERS, RPA_LIMIT, RPA_SNAPSHOTS
             (all | failures | none).
"""

import base64
import csv
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

from playwright.sync_api import expect, sync_playwright

WORKER_VERSION = "bytes-v1"

URL = os.environ.get("RPA_SITE_URL") or "https://musical-biscuit-f1d2b1.netlify.app/"
BASE_DIR = Path(os.environ.get("RPA_BASE_DIR") or ".")
CREDENTIALS_FILE = Path(os.environ.get("RPA_CREDENTIALS") or BASE_DIR / "credentials.csv")
SCREENSHOT_DIR = BASE_DIR / "screenshots"
SNAP_MODE = (os.environ.get("RPA_SNAPSHOTS") or "failures").lower()

DEFAULT_REPORT_NAME = "path_labs_report"
NAV_TIMEOUT_MS = 30_000
LOGIN_TIMEOUT_MS = 30_000
FILTER_TIMEOUT_MS = 20_000
PRINT_ENABLED_TIMEOUT_MS = 120_000
PRINT_REACTION_TIMEOUT_S = 20
PAGE_REFRESH_TIMEOUT_S = 60
RECORDS_PER_PAGE = 100
MAX_PAGES = 100

PROVIDER_VALUE = "test"
PATIENT_VALUE = "test"
LOCATION_VALUE = "test"
ENTRY_DATE_ISO = "2026-10-01"

CATEGORIES = [
    {"key": "pending", "flag": "pending_plan_completion", "tab": "Pending Plan Completion", "folder": "pending"},
    {"key": "completed", "flag": "completed", "tab": "Completed", "folder": "completed"},
    {"key": "unresolved", "flag": "unresolved", "tab": "Unresolved", "folder": "unresolved"},
]

# Injected into every frame of every page before any site script runs.
INIT_JS = """
(() => {
  window.__printCalls = 0;
  window.__printSnapshot = null;
  window.print = function () {
    window.__printCalls += 1;
    try { window.__printSnapshot = document.documentElement.outerHTML; } catch (e) {}
  };
})();
"""


class StepFailed(Exception):
    pass


def fail(msg):
    raise StepFailed(msg)


def log(msg):
    print(f"[rpa] {msg}", flush=True)


def emit(tag, obj):
    sys.stdout.write(tag + json.dumps(obj) + "\n")
    sys.stdout.flush()


class Secret(str):
    def __repr__(self):
        return "'****'"


# ----------------------------------------------------------------------------- credentials
def _truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y", "active"}


def parse_entry_date(text):
    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(text.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None


def load_credentials():
    if not CREDENTIALS_FILE.exists():
        return []
    with open(CREDENTIALS_FILE, newline="", encoding="utf-8-sig") as fh:
        rows = list(csv.DictReader(fh))
    creds, inactive, no_tabs = [], 0, []
    for row in rows:
        low = {(k or "").strip().lower(): (v or "") for k, v in row.items()}
        username = low.get("username", "").strip()
        if not username:
            continue
        if "isactive" in low and not _truthy(low["isactive"]):
            inactive += 1
            continue
        cats = [c for c in CATEGORIES if _truthy(low.get(c["flag"], ""))]
        if not cats:
            no_tabs.append(username)
            continue
        creds.append({
            "path": low.get("path", "").strip() or "/",
            "username": username,
            "password": Secret(low.get("password", "")),
            "provider": low.get("provider", "").strip() or PROVIDER_VALUE,
            "patient": low.get("patient", "").strip() or PATIENT_VALUE,
            "location": low.get("location", "").strip() or LOCATION_VALUE,
            "entry_date_iso": parse_entry_date(low.get("entrydate", "")) or ENTRY_DATE_ISO,
            "categories": cats,
        })
    wanted = os.environ.get("RPA_USERS")
    if wanted:
        names = {n.strip() for n in wanted.split(",") if n.strip()}
        creds = [c for c in creds if c["username"] in names]
    limit = os.environ.get("RPA_LIMIT")
    if limit and limit.isdigit():
        creds = creds[: int(limit)]
    log(f"worker {WORKER_VERSION} | credentials: {len(creds)} active user(s) selected, {inactive} inactive skipped"
        + (f", {len(no_tabs)} active with no tab flagged (skipped)" if no_tabs else ""))
    return creds


# ----------------------------------------------------------------------------- helpers
def safe_name(text, fallback):
    text = re.sub(r"[^\w\-. ]+", "", text or "").strip().replace(" ", "_")
    return text[:80] or fallback


def snap(page, name, force=False):
    """Screenshot to <base>/screenshots. mode all = every step; failures = only force=True; none = never."""
    if SNAP_MODE == "none" or (SNAP_MODE != "all" and not force):
        return
    try:
        SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
        (SCREENSHOT_DIR / f"{name}.png").write_bytes(page.screenshot(full_page=True))
        log(f"screenshot saved: {SCREENSHOT_DIR / (name + '.png')}")
    except Exception as exc:  # noqa: BLE001
        log(f"could not take screenshot '{name}': {exc}")


def fill_field(page, label, fallback, value):
    target = page.get_by_label(label, exact=True)
    try:
        use_label = target.count() == 1 and target.is_visible()
    except Exception:  # noqa: BLE001
        use_label = False
    field = target if use_label else fallback
    expect(field).to_be_visible()
    field.fill(value)
    expect(field).to_have_value(value)
    log(f"filled {label!r} = {value!r}")


def find_button_any_frame(page, name, timeout_ms):
    deadline = time.time() + timeout_ms / 1000
    while time.time() < deadline:
        for frame in page.frames:
            loc = frame.get_by_role("button", name=name, exact=True)
            try:
                if loc.count() > 0 and loc.first.is_visible():
                    return loc.first
            except Exception:  # noqa: BLE001
                pass
        page.wait_for_timeout(250)
    return None


def page_signature(page):
    try:
        return hash(page.locator("body").inner_text())
    except Exception:  # noqa: BLE001
        return None


def read_total_records(page):
    try:
        text = page.locator("body").inner_text()
    except Exception:  # noqa: BLE001
        return None
    patterns = [
        r"(?:showing|displaying)[^\d]{0,20}[\d,]+\s*(?:-|–|to)\s*[\d,]+\s*of\s*([\d,]+)",
        r"([\d,]+)\s+(?:records|results|entries|rows|items)\b",
        r"total[^\d]{0,20}([\d,]+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.I)
        if m:
            return int(m.group(1).replace(",", ""))
    return None


def find_page_control(page, n):
    xp = (f"xpath=//*[normalize-space(.)='{n}']"
          f"[preceding-sibling::*[normalize-space(.)='{n - 1}'] or "
          f"following-sibling::*[normalize-space(.)='{n + 1}'] or "
          f"following-sibling::*[normalize-space(.)='{n - 1}'] or "
          f"preceding-sibling::*[normalize-space(.)='{n + 1}']]")
    candidates = [page.locator(xp)]
    for role in ("button", "link"):
        candidates.append(page.get_by_role(role, name=str(n), exact=True))
    for loc in candidates:
        try:
            for i in range(loc.count()):
                el = loc.nth(i)
                if el.is_visible() and el.is_enabled():
                    return el
        except Exception:  # noqa: BLE001
            continue
    return None


def find_next_control(page):
    pat = re.compile(r"^\s*(next|>|›|»|next page)\s*$", re.I)
    for role in ("button", "link"):
        loc = page.get_by_role(role, name=pat)
        try:
            for i in range(loc.count()):
                el = loc.nth(i)
                if el.is_visible() and el.is_enabled():
                    return el
        except Exception:  # noqa: BLE001
            continue
    return None


def find_next_page(page, current):
    return find_page_control(page, current + 1) or find_next_control(page)


def wait_for_page_refresh(page, before_sig):
    deadline = time.time() + PAGE_REFRESH_TIMEOUT_S
    while time.time() < deadline:
        if page_signature(page) != before_sig:
            break
        page.wait_for_timeout(300)
    else:
        log("WARNING: page content did not visibly change after the page click")
    page.wait_for_timeout(1500)


def wait_print_cycle_start(page, label):
    btn = find_button_any_frame(page, "Print Table", 5_000)
    if btn is None:
        return
    end = time.time() + 10
    while time.time() < end:
        try:
            if btn.is_disabled():
                return
        except Exception:  # noqa: BLE001
            pass
        page.wait_for_timeout(250)
    log(f"[{label}] Print Table never went disabled after Apply Filter (fast response?)")


def wait_print_enabled(page, tag, page_no):
    print_btn = find_button_any_frame(page, "Print Table", 15_000)
    if print_btn is None:
        fail("'Print Table' button not found")
    log(f"[{tag}] page {page_no}: waiting for 'Print Table' to become enabled (up to 120s)...")
    started = time.time()
    try:
        expect(print_btn).to_be_enabled(timeout=PRINT_ENABLED_TIMEOUT_MS)
    except AssertionError:
        snap(page, f"{tag}_p{page_no}_print_table_never_enabled", force=True)
        fail(f"'Print Table' did not become enabled in time (page {page_no})")
    log(f"[{tag}] page {page_no}: 'Print Table' enabled after {time.time() - started:.1f}s")
    return print_btn


def render_pdf_bytes(page):
    page.emulate_media(media="print")
    try:
        return page.pdf(
            format="Letter", landscape=True, print_background=True, prefer_css_page_size=True,
            margin={"top": "0.4in", "bottom": "0.4in", "left": "0.4in", "right": "0.4in"},
        )
    finally:
        page.emulate_media(media="screen")   # print CSS usually hides the pager/buttons we still need


def count_records(pdf_bytes):
    """(records, declared) from the PDF bytes: dates in the first column / the 'Rows a-b of N results' line."""
    try:
        import pymupdf
    except ImportError:
        return None, None
    date_re = re.compile(r"^\d{2}/\d{2}/\d{4}$")
    rows_re = re.compile(r"Rows\s+(\d[\d,]*)\s*[-\u2013\u2014]\s*(\d[\d,]*)\s+of\s+(\d[\d,]*)", re.I)
    n, declared = 0, None
    with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
        for pg in pdf:
            n += sum(1 for w in pg.get_text("words") if w[0] < 65 and date_re.fullmatch(w[4].strip()))
            if declared is None:
                m = rows_re.search(pg.get_text("text"))
                if m:
                    f, t, tot = (int(g.replace(",", "")) for g in m.groups())
                    declared = {"from": f, "to": t, "total": tot}
    return n, declared


def login(page, cred, tag):
    path = cred["path"]
    login_url = path if path.lower().startswith("http") else urljoin(URL, path)
    log(f"[{cred['username']}] opening {login_url}")
    page.goto(login_url, wait_until="domcontentloaded")
    password = page.locator("input[type='password']")
    expect(password.first).to_be_visible(timeout=NAV_TIMEOUT_MS)
    user_by_label = page.get_by_label("Username", exact=True)
    username_field = (user_by_label if user_by_label.count() == 1
                      else page.locator("input[type='text'], input:not([type])").first)
    username_field.fill(cred["username"])
    password.first.fill(str(cred["password"]))   # never logged
    log(f"[{cred['username']}] clicking 'Sign In'")
    page.get_by_role("button", name="Sign In", exact=True).click()
    nav = page.get_by_text("Path / Labs", exact=True).first
    try:
        expect(nav).to_be_visible(timeout=LOGIN_TIMEOUT_MS)
    except AssertionError:
        snap(page, f"{tag}_login_failed", force=True)
        still = page.locator("input[type='password']").count() > 0
        fail(f"Login failed for {cred['username']}: "
             + ("sign-in form still displayed (bad credentials or wrong path)." if still
                else "'Path / Labs' did not appear after sign-in."))
    log(f"[{cred['username']}] logged in")


# ----------------------------------------------------------------------------- capture one page
def capture_report(context, page, downloads, popups, print_btn, cred, cat, ctag, page_no, multi):
    """Click Print Table; turn the site's reaction into PDF BYTES; stream them to the notebook."""
    downloads.clear()
    popups_before = len(popups)
    for frame in page.frames:
        try:
            frame.evaluate("window.__printCalls = 0")
        except Exception:  # noqa: BLE001
            pass
    snap(page, f"{ctag}_p{page_no}_before_print_click")

    log(f"[{ctag}] page {page_no}: clicking 'Print Table'")
    print_btn.click()

    outcome = None
    deadline = time.time() + PRINT_REACTION_TIMEOUT_S
    while time.time() < deadline and outcome is None:
        if downloads:
            outcome = ("download", downloads[0])
        elif len(popups) > popups_before:
            outcome = ("popup", popups[popups_before])
        else:
            for frame in page.frames:
                try:
                    if frame.evaluate("window.__printCalls || 0") > 0:
                        outcome = ("print", frame)
                        break
                except Exception:  # noqa: BLE001
                    pass
        page.wait_for_timeout(200)
    if outcome is None:
        snap(page, f"{ctag}_p{page_no}_print_no_reaction", force=True)
        fail(f"Clicking Print Table produced no download, popup or window.print() call within "
             f"{PRINT_REACTION_TIMEOUT_S}s (page {page_no}).")

    kind, obj = outcome
    log(f"[{ctag}] page {page_no}: site reaction: {kind}")

    if kind == "download":
        suggested = Path(obj.suggested_filename).stem
        pdf = Path(obj.path()).read_bytes()          # Playwright's own temp file; nothing is saved by us
    elif kind == "popup":
        obj.wait_for_load_state("load")
        obj.wait_for_timeout(500)
        suggested = obj.title()
        pdf = render_pdf_bytes(obj)
        obj.close()
    else:
        frame = obj
        suggested = page.title()
        if frame == page.main_frame:
            pdf = render_pdf_bytes(page)             # live page, exactly as the site left it
        else:
            snapshot = frame.evaluate("window.__printSnapshot") or ""
            tmp = context.new_page()
            tmp.set_content(snapshot, wait_until="load")
            pdf = render_pdf_bytes(tmp)
            tmp.close()

    if not pdf or not pdf.startswith(b"%PDF-"):
        fail(f"Captured report is not a valid PDF (page {page_no})")

    n_rows, declared = count_records(pdf)
    base = safe_name(suggested, DEFAULT_REPORT_NAME)
    emit("@@PDF@@", {
        "username": cred["username"], "category": cat["key"], "folder": cat["folder"],
        "page_no": page_no, "multi": multi, "suggested": base,
        "captured_at": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "n_records": n_rows, "declared": declared, "size": len(pdf),
        "pdf_b64": base64.b64encode(pdf).decode("ascii"),
    })
    log(f"[{ctag}] page {page_no}: report captured in memory ({len(pdf):,} bytes, {n_rows} records"
        + (f", PDF says rows {declared['from']}-{declared['to']} of {declared['total']}" if declared else "") + ")")
    return n_rows, declared


def run_category(context, page, downloads, popups, cred, cat, tag):
    username = cred["username"]
    ctag = f"{tag}_{cat['key']}"

    tab = page.get_by_role("button", name=cat["tab"], exact=True)
    expect(tab).to_be_visible()
    log(f"[{ctag}] clicking '{cat['tab']}'")
    tab.click()
    snap(page, f"{ctag}_after_tab")

    # The site keeps one filter form per tab in the DOM (inactive ones hidden): target the visible one.
    last_first = page.locator("input[placeholder='Last, First']:visible")
    expect(last_first.first).to_be_visible()
    fill_field(page, "Provider", last_first.nth(0), cred["provider"])
    fill_field(page, "Patient", last_first.nth(1), cred["patient"])
    fill_field(page, "Location", page.locator("input[placeholder='Any']:visible").first, cred["location"])
    date_input = page.locator("input[type='date']:visible").first
    expect(date_input).to_be_visible()
    date_input.fill(cred["entry_date_iso"])
    expect(date_input).to_have_value(cred["entry_date_iso"])
    log(f"[{ctag}] filled 'Entry Date' = {cred['entry_date_iso']}")
    snap(page, f"{ctag}_form_filled")

    apply_btn = find_button_any_frame(page, "Apply Filter", FILTER_TIMEOUT_MS)
    if apply_btn is None:
        snap(page, f"{ctag}_apply_filter_not_found", force=True)
        buttons = [b.inner_text().strip() for f in page.frames for b in f.locator("button").all()]
        fail(f"'Apply Filter' never appeared on '{cat['tab']}'. Buttons seen: {buttons}")
    expect(apply_btn).to_be_enabled()
    log(f"[{ctag}] clicking 'Apply Filter'")
    apply_btn.click()
    snap(page, f"{ctag}_after_apply_filter")
    wait_print_cycle_start(page, ctag)

    total_hint, multi, n_pdfs, page_no = None, False, 0, 1
    seen_rows = 0
    while True:
        print_btn = wait_print_enabled(page, ctag, page_no)
        if page_no == 1:
            total_hint = read_total_records(page)
            log(f"[{ctag}] record total shown by the site: {total_hint}")
            multi = find_page_control(page, 2) is not None or (
                total_hint is not None and total_hint > RECORDS_PER_PAGE)
            if multi:
                log(f"[{ctag}] results span several pages - capturing each page")
        n_rows, declared = capture_report(context, page, downloads, popups, print_btn, cred, cat, ctag, page_no, multi)
        n_pdfs += 1
        seen_rows += n_rows or 0
        if declared and n_rows is not None and n_rows != declared["to"] - declared["from"] + 1:
            log(f"WARNING [{ctag}] page {page_no}: found {n_rows} records, the PDF declares "
                f"{declared['to'] - declared['from'] + 1}")

        nxt = find_next_page(page, page_no)
        if nxt is None:
            if total_hint and page_no * RECORDS_PER_PAGE < total_hint:
                snap(page, f"{ctag}_pagination_not_found", force=True)
                fail(f"Site reports {total_hint} records but no control for page {page_no + 1} was found.")
            break
        if page_no >= MAX_PAGES:
            fail(f"Stopped at MAX_PAGES={MAX_PAGES}")
        log(f"[{ctag}] going to page {page_no + 1}")
        before = page_signature(page)
        nxt.click()
        page_no += 1
        wait_for_page_refresh(page, before)
        snap(page, f"{ctag}_p{page_no}_after_page_change")

    log(f"[{ctag}] DONE - {n_pdfs} report page(s), {seen_rows} records")
    return n_pdfs


def run_user(browser, cred):
    username = cred["username"]
    tag = safe_name(username, "user")
    context = browser.new_context(accept_downloads=True, viewport={"width": 1600, "height": 1000})
    context.add_init_script(INIT_JS)
    page = context.new_page()
    page.set_default_timeout(NAV_TIMEOUT_MS)
    popups, downloads = [], []
    context.on("page", lambda p: popups.append(p) if p != page else None)
    page.on("download", lambda d: downloads.append(d))
    page.on("dialog", lambda d: d.accept())

    errors = {}
    try:
        login(page, cred, tag)
        snap(page, f"{tag}_after_login")
        log(f"[{username}] clicking 'Path / Labs'")
        page.get_by_text("Path / Labs", exact=True).first.click()
        path_btn = page.get_by_role("button", name="Path", exact=True)
        expect(path_btn).to_be_visible()
        log(f"[{username}] clicking 'Path'")
        path_btn.click()
        log(f"[{username}] tabs to download: {[c['tab'] for c in cred['categories']]}")
        for cat in cred["categories"]:
            try:
                run_category(context, page, downloads, popups, cred, cat, tag)
            except Exception as exc:  # keep going with the next tab
                errors[cat["key"]] = f"{type(exc).__name__}: {str(exc)[:300]}"
                log(f"[{username}] '{cat['tab']}' FAILED: {errors[cat['key']]}")
                snap(page, f"{tag}_{cat['key']}_failure", force=True)
    except Exception as exc:
        errors["login_or_navigation"] = f"{type(exc).__name__}: {str(exc)[:300]}"
        log(f"[{username}] FAILED: {errors['login_or_navigation']}")
        snap(page, f"{tag}_failure", force=True)
    finally:
        context.close()
    emit("@@USER@@", {"username": username, "status": "failed" if errors else "ok", "errors": errors})
    return not errors


def main():
    creds = load_credentials()
    if not creds:
        log("No active users with a flagged tab - nothing to do.")
        emit("@@DONE@@", {"users": 0, "failed": 0})
        return 0
    failed = 0
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        try:
            for cred in creds:
                if not run_user(browser, cred):
                    failed += 1
        finally:
            browser.close()
    emit("@@DONE@@", {"users": len(creds), "failed": failed})
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())
'''

In [0]:
import base64
import csv
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

# category key -> (sub-folder, suffix for CSV / table names)
CATEGORY_FOLDERS = {
    "pending": ("pending", "_pend"),
    "completed": ("completed", "_comp"),
    "unresolved": ("unresolved", "_unres"),
}

DEFAULT_BOUNDS = [31.5, 71.9, 130.6, 200.3, 257.8, 339.5, 404.1, 452.6, 531.6, 601.9, 663.7, 692.1, 739.9]  # fallback header x0s
COLUMN_NAMES = ["result_date", "patient_raw", "facility", "location", "result", "actions_and_plans",
                "comments_assigned_to", "flags", "notifications", "notes", "patient_portal",
                "results_processed", "photos"]
# first word of every column header, in left-to-right order
HEADER_FIRST_WORDS = ["Date", "Patient", "Facility", "Location", "Result", "Actions", "Comments",
                      "Flags", "Notifications", "Notes", "Patient", "Results", "Photos"]
X_TOL = 3.0

DATE_RE = re.compile(r"^\d{2}/\d{2}/\d{4}$")
DOB_RE = re.compile(r"\bDOB\s+(\d{2}/\d{2}/\d{4})\b")
MRN_RE = re.compile(r"\bMRN\s+([A-Z]{2,}-\d+)\b")
ROWS_RE = re.compile(r"Rows\s+(\d[\d,]*)\s*[-\u2013\u2014]\s*(\d[\d,]*)\s+of\s+(\d[\d,]*)", re.I)


def clean_text(v):
    if not v:
        return None
    v = re.sub(r"\s+", " ", v).strip()
    return v or None


def join_lines(lines):
    """Join wrapped lines; a line ending in a glued hyphen ('re-', 'NF-', '5-') continues without a space."""
    out = ""
    for ln in (clean_text(l) for l in lines):
        if not ln:
            continue
        if out and re.search(r"\S-$", out):
            out += ln
        else:
            out = f"{out} {ln}" if out else ln
    return out or None


def words_to_lines(words, tol=2.5):
    lines = []
    for w in sorted(words, key=lambda w: (w[1], w[0])):
        for ln in lines:
            if abs(ln["y"] - w[1]) <= tol:
                ln["w"].append(w); break
        else:
            lines.append({"y": w[1], "w": [w]})
    for ln in lines:
        ln["w"].sort(key=lambda w: w[0])
    return sorted(lines, key=lambda l: l["y"])


def column_lines(words, x_left, x_right):
    sel = [w for w in words if x_left <= w[0] < x_right]
    return [" ".join(w[4] for w in ln["w"]) for ln in words_to_lines(sel)]


def find_header_bounds(words):
    """
    Locate the table header row ('Date  Patient  Facility ...') and return
    (left x of each of the 13 columns, y of the header row's bottom line) or None.
    """
    dates = [w for w in words if w[4] == "Date" and w[0] < 80]
    for d in sorted(dates, key=lambda w: w[1]):
        # header block = this line plus the wrapped line above it
        block = [w for w in words if d[1] - 14 <= w[1] <= d[1] + 3]
        if not any(w[4] == "Facility" for w in block):
            continue
        block.sort(key=lambda w: (w[0], w[1]))
        xs, last = [], -1.0
        for first in HEADER_FIRST_WORDS:
            hit = next((w for w in block if w[4] == first and w[0] > last + 1), None)
            if hit is None:
                break
            xs.append(hit[0]); last = hit[0]
        if len(xs) == len(HEADER_FIRST_WORDS):
            return xs, max(w[3] for w in block)
    return None


def drop_netlify_badge(words):
    """Remove the floating 'Powered by Netlify' badge text that is printed on page 1."""
    ws = sorted(words, key=lambda w: (round(w[1]), w[0]))
    bad = set()
    for i, w in enumerate(ws):
        if w[4] == "Powered" and i + 2 < len(ws) and ws[i + 1][4] == "by" and ws[i + 2][4] == "Netlify":
            bad.update((id(ws[i]), id(ws[i + 1]), id(ws[i + 2])))
    return [w for w in words if id(w) not in bad]


def parse_declared_rows(text):
    m = ROWS_RE.search(text or "")
    if not m:
        return None
    f = [int(g.replace(",", "")) for g in m.groups()]
    return {"from": f[0], "to": f[1], "total": f[2]}


def extract_pdf_records(source):
    """
    source = PDF bytes (in memory) or a file path.
    -> (records, declared) ; declared = {'from','to','total'} from the 'Rows a-b of N results' line.
    """
    import pymupdf
    records, declared, prev_bounds = [], None, None
    doc = (pymupdf.open(stream=bytes(source), filetype="pdf")
           if isinstance(source, (bytes, bytearray)) else pymupdf.open(str(source)))
    with doc as pdf:
        for page_no, page in enumerate(pdf, start=1):
            words = drop_netlify_badge(page.get_text("words", sort=True))
            if declared is None:
                declared = parse_declared_rows(page.get_text("text"))

            hdr = find_header_bounds(words)
            if hdr:
                xs, header_bottom = hdr
                prev_bounds = xs
            else:                        # continuation page without a header: reuse the last bounds
                xs, header_bottom = (prev_bounds or DEFAULT_BOUNDS), 0.0
            lefts = [x - X_TOL for x in xs]
            edges = lefts + [page.rect.width + 1]
            edges[0] = 0.0

            body = [w for w in words if w[1] >= header_bottom - 0.5]
            anchors = sorted(
                ((w[1], w[4].strip()) for w in body if w[0] < edges[1] and DATE_RE.fullmatch(w[4].strip())),
                key=lambda a: a[0])
            page_bottom = page.rect.height + 1

            for i, (y0, date_text) in enumerate(anchors):
                y1 = anchors[i + 1][0] - 4 if i + 1 < len(anchors) else page_bottom
                rec_words = [w for w in body if y0 - 4 <= w[1] < y1]
                cols = {name: column_lines(rec_words, edges[c], edges[c + 1]) for c, name in enumerate(COLUMN_NAMES)}

                row = {n: join_lines(l) for n, l in cols.items()}
                row["result_date"] = date_text
                # notifications: keep separate entries apart ("Free Form - ..." starts a new one)
                notif = join_lines(cols["notifications"])
                if notif:
                    parts = [p.strip() for p in re.split(r"\s(?=Free Form\s-\s)", notif) if p.strip()]
                    row["notifications"] = " | ".join(parts)
                patient_raw = row.get("patient_raw") or ""
                dm, mm = DOB_RE.search(patient_raw), MRN_RE.search(patient_raw)
                row["dob"] = dm.group(1) if dm else None
                row["mrn"] = mm.group(1) if mm else None
                name = MRN_RE.sub("", DOB_RE.sub("", patient_raw))
                row["patient_name"] = clean_text(name)
                row["source_page"] = str(page_no)
                records.append(row)
    return records, declared


RECORD_COLUMNS = [
    "result_date", "patient_name", "dob", "mrn", "facility", "location", "result",
    "actions_and_plans", "comments_assigned_to", "flags", "notifications", "notes",
    "patient_portal", "results_processed", "photos", "source_page",
]
FILE_COLUMNS = ["category", "source_file", "username", "downloaded_at", "report_page", "record_seq",
                "declared_from", "declared_to", "declared_total"]
ALL_COLUMNS = RECORD_COLUMNS + FILE_COLUMNS


def report_to_rows(meta, pdf_bytes):
    """Parse one report (PDF bytes) -> (row tuples in ALL_COLUMNS order, declared, logical file name)."""
    recs, declared = extract_pdf_records(pdf_bytes)
    name = f"{meta['suggested']}_{meta['username']}_{meta['captured_at']}"
    if meta.get("multi"):
        name += f"_p{meta['page_no']}"
    name += ".pdf"                                          # logical name only - nothing is stored under it
    d = declared or {}
    rows = [
        tuple(rec.get(c) for c in RECORD_COLUMNS)
        + (meta["category"], name, meta["username"], meta["captured_at"], str(meta["page_no"]), str(seq),
           str(d["from"]) if d else None, str(d["to"]) if d else None, str(d["total"]) if d else None)
        for seq, rec in enumerate(recs, start=1)
    ]
    return rows, declared, name

In [0]:
def _truthy(v):
    return str(v).strip().lower() in {"1", "true", "yes", "y", "active"}


def check_credentials_header(path):
    with open(path, newline="", encoding="utf-8-sig") as fh:
        header = [(h or "").strip().lower() for h in next(csv.reader(fh))]
    need = ["path", "username", "password", "isactive", "pending_plan_completion", "completed"]
    missing = [c for c in need if c not in header]
    print(f"credentials.csv columns: {len(header)} | missing required: {missing or 'none'}")
    if "isactive" in missing:
        print("  WARNING: no isActive column -> every user is treated as active")
    if "pending_plan_completion" in missing and "completed" in missing:
        raise RuntimeError("credentials.csv has neither Pending_plan_completion nor Completed - nothing to download.")


if not os.path.isfile(CREDENTIALS_FILE):
    raise FileNotFoundError(f"{CREDENTIALS_FILE} not found (upload credentials.csv as a file into base_dir).")
check_credentials_header(CREDENTIALS_FILE)

# Counts only - no credential values are printed.
with open(CREDENTIALS_FILE, newline="", encoding="utf-8-sig") as fh:
    _rows = [{(k or "").strip().lower(): (v or "") for k, v in r.items()} for r in csv.DictReader(fh)]
_rows = [r for r in _rows if r.get("username", "").strip()]
_active = [r for r in _rows if "isactive" not in r or _truthy(r["isactive"])]
if USERS:
    _w = {u.strip() for u in USERS.split(",") if u.strip()}
    _active = [r for r in _active if r["username"].strip() in _w]
if LIMIT.isdigit():
    _active = _active[: int(LIMIT)]
print(f"Users in credentials.csv: {len(_rows)} | active & selected: {len(_active)}")
print("  tabs -> pending:", sum(_truthy(r.get("pending_plan_completion", "")) for r in _active),
      "| completed:", sum(_truthy(r.get("completed", "")) for r in _active),
      "| unresolved:", sum(_truthy(r.get("unresolved", "")) for r in _active))

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DOWNLOAD_DIR = os.path.join(BASE_DIR, "downloads")
RUN_DIR = os.path.join(DOWNLOAD_DIR, RUN_ID)

# The worker script goes to the driver's LOCAL temp disk (not a workspace location).
worker_path = os.path.join(tempfile.gettempdir(), f"rpa_worker_{RUN_ID}.py")
with open(worker_path, "w", encoding="utf-8") as fh:
    fh.write(WORKER_SOURCE)

env = {
    **os.environ,                       # includes LD_LIBRARY_PATH set by the Chromium setup cell
    "RPA_BASE_DIR": BASE_DIR,
    "RPA_CREDENTIALS": CREDENTIALS_FILE,
    "RPA_SITE_URL": SITE_URL,
    "RPA_SNAPSHOTS": SNAPSHOTS,
    "PYTHONUNBUFFERED": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
}
if USERS:
    env["RPA_USERS"] = USERS
if LIMIT:
    env["RPA_LIMIT"] = LIMIT

all_rows, report_log, user_status, parse_errors = [], [], [], []
print(f"\nStarting the browser worker (run {RUN_ID}); reports are parsed as they arrive...\n")
# stderr goes to its own temp file: native-library warnings written to stderr would otherwise be
# interleaved into the middle of the long @@PDF@@ lines on stdout and corrupt their JSON.
stderr_fh = tempfile.TemporaryFile(mode="w+", encoding="utf-8")
proc = subprocess.Popen([sys.executable, worker_path], cwd=tempfile.gettempdir(), env=env,
                        stdout=subprocess.PIPE, stderr=stderr_fh, text=True, bufsize=1)
for line in proc.stdout:
    if line.startswith("@@PDF@@"):
        meta = json.loads(line[len("@@PDF@@"):])
        pdf_bytes = base64.b64decode(meta.pop("pdf_b64"))               # the report, as bytes
        try:
            rows, declared, name = report_to_rows(meta, pdf_bytes)
        except Exception as exc:  # noqa: BLE001
            parse_errors.append((meta["username"], meta["category"], meta["page_no"], f"{type(exc).__name__}: {exc}"))
            print(f"  [parse ERROR] {meta['username']} {meta['category']} page {meta['page_no']}: {exc}")
            continue
        all_rows += rows
        expected = (declared["to"] - declared["from"] + 1) if declared else None
        report_log.append((meta["username"], meta["category"], meta["page_no"], len(rows), expected))
        flag = "" if expected in (None, len(rows)) else f"  <-- MISMATCH (PDF declares {expected})"
        print(f"  [parsed] {meta['username']} | {meta['category']} | page {meta['page_no']}: {len(rows)} rows{flag}")
        if SAVE_PDFS:                                                   # optional audit copy
            out = os.path.join(RUN_DIR, CATEGORY_FOLDERS.get(meta["category"], (meta["category"], ""))[0])
            os.makedirs(out, exist_ok=True)
            with open(os.path.join(out, name), "wb") as fh:
                fh.write(pdf_bytes)
    elif line.startswith("@@USER@@"):
        user_status.append(json.loads(line[len("@@USER@@"):]))
    elif line.startswith("@@DONE@@"):
        pass
    else:
        print(line, end="")
rc = proc.wait()
os.remove(worker_path)
stderr_fh.seek(0)
_err = stderr_fh.read().strip()
stderr_fh.close()
if _err:
    print("\nworker stderr:\n" + _err[-5000:])

print(f"\nworker exit code: {rc}  (0 = all users ok, 1 = some users/tabs failed)")
print(f"reports parsed: {len(report_log)} | rows: {len(all_rows)} | parse errors: {len(parse_errors)}")
for st in user_status:
    if st["status"] != "ok":
        print(f"  FAILED user {st['username']}: {st['errors']}")
if not all_rows:
    raise RuntimeError("No rows were extracted - see the worker log above.")

credentials.csv columns: 11 | missing required: none
Users in credentials.csv: 51 | active & selected: 10
  tabs -> pending: 10 | completed: 10 | unresolved: 0

Starting the browser worker (run 20261103_074819); reports are parsed as they arrive...

[rpa] worker bytes-v1 | credentials: 10 active user(s) selected, 41 inactive skipped
[rpa] [user_a0] opening https://musical-biscuit-f1d2b1.netlify.app/a0/
[rpa] [user_a0] clicking 'Sign In'
[rpa] [user_a0] logged in
[rpa] [user_a0] clicking 'Path / Labs'
[rpa] [user_a0] clicking 'Path'
[rpa] [user_a0] tabs to download: ['Pending Plan Completion', 'Completed']
[rpa] [user_a0_pending] clicking 'Pending Plan Completion'
[rpa] filled 'Provider' = 'test'
[rpa] filled 'Patient' = 'test'
[rpa] filled 'Location' = 'test'
[rpa] [user_a0_pending] filled 'Entry Date' = 2026-10-01
[rpa] [user_a0_pending] clicking 'Apply Filter'
[rpa] [user_a0_pending] Print Table never went disabled after Apply Filter (fast response?)
[rpa] [user_a0_pending] page 1: w

In [0]:
schema = StructType([StructField(c, StringType(), True) for c in ALL_COLUMNS])
raw_df = spark.createDataFrame(all_rows, schema)

pathology_df = (
    raw_df
    .withColumn("result_date", F.to_date("result_date", "MM/dd/yyyy"))
    .withColumn("dob", F.to_date("dob", "MM/dd/yyyy"))
    .withColumn("downloaded_at", F.to_timestamp("downloaded_at", "yyyyMMdd_HHmmss"))
    .withColumn("report_page", F.col("report_page").cast("int"))
    .withColumn("source_page", F.col("source_page").cast("int"))
    .withColumn("record_seq", F.col("record_seq").cast("int"))
    .withColumn("declared_from", F.col("declared_from").cast("int"))
    .withColumn("declared_to", F.col("declared_to").cast("int"))
    .withColumn("declared_total", F.col("declared_total").cast("int"))
    .cache()
)
display(pathology_df.limit(10))

result_date,patient_name,dob,mrn,facility,location,result,actions_and_plans,comments_assigned_to,flags,notifications,notes,patient_portal,results_processed,photos,source_page,category,source_file,username,downloaded_at,report_page,record_seq,declared_from,declared_to,declared_total
2026-10-12,"Chen, Kenji I.",1994-01-12,NF-180060,Northfield Dermatology & Skin Surgery Center,"A: right shin Seborrheic Keratosis,","Reassure: irritated benign, further treatment",no,BENIGN,"Free Okafor, 10/14/26 PM",Form - Priya - 03:30,Release,10/13/2026 03:42 FINAL,PM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,1,1,100,250
2026-10-12,"Donovan, Daniel J.",1989-04-05,NF-165073,Northfield Dermatology & Skin Surgery Center,A: mid upper Squamous back Carcinoma,Cell Schedule: in situ visit recheck and options,SRT VS to VS MOHS site discuss,EXC CANCEROUS,"Free Nguyen, 10/15/26 PM",Form - Tara - 04:24,Hide,10/13/2026 04:50 FINAL,PM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,2,1,100,250
2026-10-11,"Underwood, Edward V.",1982-10-20,NF-157746,Northfield Dermatology & Skin Surgery Center,"A: right nasal Basal ala Carcinoma, type",Cell Schedule: nodular Mohs consult,MOHS surgery Lindqvist,- Dr. CANCEROUS,"Free Jordan 03:32 Free Brooks, 10/18/26 AM","Form - Hale, - 10/16/26 PM Form - Samir - 09:46",Hide,10/14/2026 05:08 FINAL,PM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,3,1,100,250
2026-10-10,"Marchetti, Olivia O.",1965-06-12,NF-235852,Northfield Dermatology & Skin Surgery Center,A: right shin Atypical Intraepidermal Melanocytic Proliferation,Schedule: excision,re- RE-EXC Hale,- ATYPICAL,Free Jordan 02:18,"Form - Hale, Call - 10/12/26 (HIPAA PM",daughter Hide on file),10/11/2026 08:15 FINAL,AM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,4,1,100,250
2026-10-09,"Rasmussen, Ingrid X.",1942-02-05,NF-202803,Northfield Dermatology & Skin Surgery Center,"A: left Basal preauricular Carcinoma, cheek superficial",Cell Schedule: ED&C type topical,ED&C vs FU 5-FU,VS 5- CANCEROUS,"Free Ramos, 10/15/26 PM",Form - Elena - 02:29,Hide,10/12/2026 05:04 FINAL,PM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,5,1,100,250
2026-10-08,"Sandoval, Walter R.",1993-01-05,NF-249385,Northfield Dermatology & Skin Surgery Center,A: right upper Epidermal arm Cyst,Inclusion Reassure: benign,null,BENIGN,"Free Okafor, 10/11/26 PM",Form - Priya - 04:59,Release,10/10/2026 03:24 FINAL,PM,null,1,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,6,1,100,250
2026-10-08,"Fitzgerald, Howard X.",1938-10-19,NF-222142,Northfield Dermatology & Skin Surgery Center,A: left inferior Atypical central malar Intraepidermal cheek Melanocytic Proliferation,Schedule: excision,re- RE-EXC Hale,- ATYPICAL,Free Jordan 10:17,"Form - Hale, - 10/12/26 AM",Hide,10/11/2026 05:05 FINAL,PM,null,2,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,7,1,100,250
2026-10-08,"Abernathy, Priscilla C.",1965-06-04,NF-204402,Northfield Dermatology & Skin Surgery Center,"A: right dorsal Basal index finger Carcinoma, type",Cell Schedule: nodular Mohs consult,MOHS surgery Lindqvist,- Dr. CANCEROUS,"Free Nguyen, 10/12/26 PM Free Ramos, 10/14/26 AM",Form - Tara - 03:59 Form - Elena - 08:21,Hide,10/10/2026 04:57 FINAL,PM,null,2,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,8,1,100,250
2026-10-07,"Sandoval, Teresa F.",1989-10-24,NF-208058,Northfield Dermatology & Skin Surgery Center,A: chin Hypertrophic Keratosis,Actinic Reassure: treat LN2 visit,LN2 NEXT with VISIT at next,PRECANCEROUS,"Free Ramos, 10/09/26 AM",Form - Elena - 08:22,Release,10/08/2026 11:50 FINAL,AM,null,2,pending,Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,2026-11-03T07:48:33Z,1,9,1,100,250
2026-10-05,"Sandoval, Nathan V.",

In [0]:
CSV_COLUMNS = RECORD_COLUMNS + ["category", "username", "downloaded_at", "report_page", "record_seq", "source_file"]


def _fmt(v):
    if v is None:
        return ""
    if hasattr(v, "isoformat"):
        return v.isoformat(sep=" ") if hasattr(v, "hour") else v.isoformat()
    return str(v)


def write_csv(dest, rows):
    """Build the CSV on local disk, then copy it to the (FUSE) workspace folder."""
    fd, tmp = tempfile.mkstemp(suffix=".csv")
    os.close(fd)
    with open(tmp, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(CSV_COLUMNS)
        for r in rows:
            w.writerow([_fmt(r[c]) for c in CSV_COLUMNS])
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copyfile(tmp, dest)
    os.remove(tmp)


if SAVE_CSV:
    collected = pathology_df.orderBy("category", "source_file", "record_seq").collect()   # <= 100 rows per report
    written = []
    for folder, (cat_key, suffix) in CATEGORY_FOLDERS.items():
        rows_c = [r for r in collected if r["category"] == cat_key]
        if not rows_c:
            continue
        out_dir = os.path.join(RUN_DIR, folder)
        by_file, by_user = {}, {}
        for r in rows_c:
            by_file.setdefault(r["source_file"], []).append(r)
            by_user.setdefault(r["username"], []).append(r)

        for fname, rows in by_file.items():                      # 1) one CSV per report page
            dest = os.path.join(out_dir, os.path.splitext(fname)[0] + suffix + ".csv")
            write_csv(dest, rows)
            written.append(dest)

        for user, rows in by_user.items():                       # 2) merged CSV per user (only if several pages)
            files = sorted({r["source_file"] for r in rows})
            if len(files) > 1:
                first_stem = re.sub(r"_p\d+$", "", os.path.splitext(files[0])[0])
                rows_sorted = sorted(rows, key=lambda r: (r["report_page"] or 0, r["record_seq"] or 0))
                dest = os.path.join(out_dir, f"{first_stem}_all{suffix}.csv")
                write_csv(dest, rows_sorted)
                written.append(dest)

        dest = os.path.join(out_dir, f"pathology_all_users_{RUN_ID}{suffix}.csv")   # 3) everything of this category
        write_csv(dest, rows_c)
        written.append(dest)
        print(f"{folder:11}: {len(rows_c)} rows -> {out_dir}")
    print(f"{len(written)} CSV file(s) written")
else:
    print("save_csv = no - skipping CSV files")

save_csv = no - skipping CSV files


In [0]:
for folder, (cat_key, suffix) in CATEGORY_FOLDERS.items():
    df_c = pathology_df.filter(F.col("category") == cat_key)
    if df_c.limit(1).count() == 0:
        continue
    print("=" * 100)
    print(f"CATEGORY: {cat_key}  (suffix {suffix})  - extracted rows: {df_c.count()}")
    print("=" * 100)

    # Every report is compared with the 'Rows a-b of N results' line printed inside it.
    per_file = (
        df_c.groupBy("source_file", "username", "report_page", "declared_from", "declared_to", "declared_total")
        .count()
        .withColumn("expected", F.col("declared_to") - F.col("declared_from") + 1)
        .withColumn("ok", F.col("count") == F.col("expected"))
        .orderBy("username", "report_page", "source_file")
    )
    display(per_file)
    bad = per_file.filter(F.col("ok") == False)       # noqa: E712
    print("Reports whose extracted row count differs from the count printed in the PDF:", bad.count())
    display(bad)

    # The pages of one user should add up to the declared total.
    display(
        per_file.groupBy("username").agg(
            F.sum("count").alias("rows_extracted"),
            F.max("declared_total").alias("declared_total"),
            F.count("*").alias("report_pages"),
        ).withColumn("ok", F.col("rows_extracted") == F.col("declared_total")).orderBy("username")
    )

    print("Rows with missing patient_name, dob or mrn:")
    display(
        df_c.filter(F.col("patient_name").isNull() | F.col("dob").isNull() | F.col("mrn").isNull())
        .select("source_file", "result_date", "patient_name", "dob", "mrn", "source_page")
    )

    print("Duplicate MRNs inside one user's extract:")
    display(df_c.groupBy("username", "mrn").count().filter(F.col("count") > 1).orderBy(F.desc("count")))

    display(
        df_c.select(
            "source_file", "result_date", "patient_name", "dob", "mrn", "location", "result",
            "actions_and_plans", "comments_assigned_to", "flags", "notifications",
            "notes", "patient_portal", "results_processed",
        ).orderBy(F.col("result_date").desc()).limit(25)
    )

    required_columns = ["result_date", "patient_name", "dob", "mrn", "result", "flags", "results_processed"]
    display(
        df_c.select([
            F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(f"{c}_missing")
            for c in required_columns
        ])
    )

CATEGORY: pending  (suffix _pend)  - extracted rows: 2500


source_file,username,report_page,declared_from,declared_to,declared_total,count,expected,ok
Northfield_EMR_user_a0_20261103_074833_p1.pdf,user_a0,1,1,100,250,100,100,true
Northfield_EMR_user_a0_20261103_074835_p2.pdf,user_a0,2,101,200,250,100,100,true
Northfield_EMR_user_a0_20261103_074837_p3.pdf,user_a0,3,201,250,250,50,50,true
Northfield_EMR_user_a1_20261103_074901_p1.pdf,user_a1,1,1,100,250,100,100,true
Northfield_EMR_user_a1_20261103_074903_p2.pdf,user_a1,2,101,200,250,100,100,true
Northfield_EMR_user_a1_20261103_074905_p3.pdf,user_a1,3,201,250,250,50,50,true
Northfield_EMR_user_a2_20261103_074930_p1.pdf,user_a2,1,1,100,250,100,100,true
Northfield_EMR_user_a2_20261103_074932_p2.pdf,user_a2,2,101,200,250,100,100,true
Northfield_EMR_user_a2_20261103_074934_p3.pdf,user_a2,3,201,250,250,50,50,true
Northfield_EMR_user_a3_20261103_074958_p1.pdf,user_a3,1,1,100,250,100,100,true


Reports whose extracted row count differs from the count printed in the PDF: 0


source_file,username,report_page,declared_from,declared_to,declared_total,count,expected,ok


username,rows_extracted,declared_total,report_pages,ok
user_a0,250,250,3,true
user_a1,250,250,3,true
user_a2,250,250,3,true
user_a3,250,250,3,true
user_a4,250,250,3,true
user_a5,250,250,3,true
user_a6,250,250,3,true
user_a7,250,250,3,true
user_a8,250,250,3,true
user_a9,250,250,3,true


Rows with missing patient_name, dob or mrn:


... (12 outputs left out)

Duplicate MRNs inside one user's extract:


username,mrn,count


source_file,result_date,patient_name,dob,mrn,location,result,actions_and_plans,comments_assigned_to,flags,notifications,notes,patient_portal,results_processed
Northfield_EMR_user_a6_20261103_075141_p1.pdf,2026-10-12,"Abernathy, Maria T.",1938-10-26,NF-176325,A: right Hypertrophic earlobe Keratosis,Actinic Reassure: treat LN2 visit,LN2 NEXT with VISIT at next,PRECANCEROUS,"Free Okafor, 10/14/26 AM",Form - On Priya - anticoagulation 10:48,Release,10/13/2026 11:25 FINAL,AM
Northfield_EMR_user_a1_20261103_074916_p1.pdf,2026-10-12,"Quinlan, Samuel P.",1997-11-07,NF-209661,"A: nasal tip Basal Carcinoma, infiltrative",Cell Schedule: Mohs type,MOHS surgery Lindqvist,- Dr. CANCEROUS,"Free Ramos, 10/15/26 AM",Form - Elena - 09:41,Hide,10/14/2026 02:50 FINAL,PM
Northfield_EMR_user_a3_20261103_075013_p1.pdf,2026-10-12,"Eriksen, Lillian P.",1984-09-25,NF-184207,"A: right Basal medial calf Carcinoma, superficial",Cell Schedule: ED&C type topical,ED&C vs FU 5-FU,VS 5- CANCEROUS,"Free Okafor, 10/16/26 PM",Form - Priya - 02:11,Hide,10/13/2026 01:23 FINAL,PM
Northfield_EMR_user_a0_20261103_074848_p1.pdf,2026-10-12,"Novak, Kenji W.",1952-06-19,NF-172348,A: mid forehead,Epidermal Inclusion Cyst,Reassure: benign,BENIGN,null,"Free Form - Hale, Jordan - 10/16/26 02:01 PM",Hx of organ Release transplant,10/14/2026 09:14 FINAL,AM
Northfield_EMR_user_a8_20261103_075238_p1.pdf,2026-10-12,"Lindgren, Curtis N.",1958-12-11,NF-240111,A: left dorsal Dysplastic index finger with margins,"Nevus Reassure: mild atypia, margins clear",clear,ATYPICAL,"Free Ramos, 10/14/26 AM Free Nguyen, 10/15/26 AM",Form - Pt Elena - appt 09:15 Form - Tara - 11:33,prefers AM Hide,10/13/2026 05:29 FINAL,PM
Northfield_EMR_user_a2_20261103_074945_p1.pdf,2026-10-12,"Thornton, Ingrid H.",1984-11-20,NF-156676,A: right upper back,"Basal Cell Carcinoma, superficial type",Schedule: ED&C ED&C vs FU topical 5-FU,VS 5- CANCEROUS,null,"Free Form - Hale, Jordan - 10/18/26 08:17 AM | Free Form - Okafor, Priya - 10/21/26 01:54 PM",Hide,10/15/2026 04:45 FINAL,PM
Northfield_EMR_user_a5_20261103_075112_p1.pdf,2026-10-12,"Thornton, Omar U.",1999-10-09,NF-234496,A: right lateral Dysplastic thigh with margins,"Nevus Reassure: mild atypia, margins clear",clear,ATYPICAL,"Free Okafor, 10/15/26 PM",Form - Priya - 12:36,Hide,10/13/2026 01:55 FINAL,PM
Northfield_EMR_user_a2_20261103_074945_p1.pdf,2026-10-12,"Oduya, Andre M.",1980-04-07,NF-199830,A: right medial calf,Atypical Intraepidermal Melanocytic Proliferation,Schedule: re- RE-EXC excision Hale,- ATYPICAL,null,"Free Form - Hale, Jordan - 10/16/26 03:19 PM",Hide,10/14/2026 12:38 FINAL,PM
Northfield_EMR_user_a7_20261103_075210_p1.pdf,2026-10-12,"Moreau, Helen J.",1991-06-22,NF-220808,A: upper lip,"Basal Cell Schedule: Carcinoma, nodular Mohs type consult",MOHS surgery Lindqvist,- Dr. CANCEROUS,"Free Nguyen, 10/15/26 PM",Form - Tara - 04:17,Hide,10/13/2026 11:06 FINAL,AM
Northfield_EMR_user_a4_20261103_075043_p1.pdf,2026-10-12,"Eriksen, Frank F.",1998-08-02,NF-165639,A: mid upper No back Squamous Carcinoma,Residual Reassure: Cell margins in situ,clear,CANCEROUS,"Free Nguyen, 10/14/26 PM",Form - Tara - 01:10,Hide,10/13/2026 08:35 FINAL,AM


result_date_missing,patient_name_missing,dob_missing,mrn_missing,result_missing,flags_missing,results_processed_missing
0,0,0,0,0,499,0


In [0]:
USER_TABLE_PREFIX = "inmem_"     # per-user tables: <catalog>.<schema>.inmem_<username><suffix>, e.g. inmem_user_a1_comp


def _safe_name(v):
    """Lower-case and replace anything that is not a letter, digit or underscore (valid table identifier)."""
    return re.sub(r"[^a-z0-9_]", "_", str(v).strip().lower())


if OUTPUT_TABLE:
    # per-user tables go to the same catalog.schema as the combined output_table
    _parts = OUTPUT_TABLE.split(".")
    TABLE_NAMESPACE = ".".join(_parts[:-1])
    if len(_parts) != 3:
        print(f"WARNING: output_table '{OUTPUT_TABLE}' is not catalog.schema.table - per-user tables use the current schema")

    for folder, (cat_key, suffix) in CATEGORY_FOLDERS.items():
        df_c = pathology_df.filter(F.col("category") == cat_key)
        if df_c.limit(1).count() == 0:
            continue

        # 1) combined table for this category (all users)
        table_name = f"{OUTPUT_TABLE}{suffix}"
        df_c.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
        print(f"[{cat_key}] combined: saved {df_c.count()} rows to {table_name}")

        # 2) one table per user for this category
        users_c = sorted(r["username"] for r in df_c.select("username").distinct().collect() if r["username"])
        for user in users_c:
            df_u = df_c.filter(F.col("username") == user)
            user_table = f"{USER_TABLE_PREFIX}{_safe_name(user)}{suffix}"
            if TABLE_NAMESPACE:
                user_table = f"{TABLE_NAMESPACE}.{user_table}"
            df_u.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(user_table)
            print(f"[{cat_key}]   user {user}: saved {df_u.count()} rows to {user_table}")
else:
    print("No output_table set - skipping write.")

[pending] combined: saved 2500 rows to us_scdb_phi_dev.comm_bronze.inmem_pathology_results_pend
[pending]   user user_a0: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a0_pend
[pending]   user user_a1: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a1_pend
[pending]   user user_a2: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a2_pend
[pending]   user user_a3: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a3_pend
[pending]   user user_a4: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a4_pend
[pending]   user user_a5: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a5_pend
[pending]   user user_a6: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a6_pend
[pending]   user user_a7: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a7_pend
[pending]   user user_a8: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a8_pend
[pending]   user user_a9: saved 250 rows to us_scdb_phi_dev.comm_bronze.inmem_user_a9